# 🚀 Fine-Tuning Qwen Models for Indian SMS Entity Extraction

**Production-grade Small Language Model (SLM) Fine-Tuning Guide**

This notebook demonstrates how to fine-tune **Qwen3.5 2B** (or Qwen2.5 1.5B/3B) using **LoRA (Low-Rank Adaptation)** and Hugging Face's **TRL (Transformer Reinforcement Learning / SFTTrainer)** to extract structured, validated JSON entities from messy, unstructured Indian SMS messages.

### 🎯 The Challenge
Indian SMS messages are highly unstructured, containing diverse sender IDs (`VM-HDFCBK`, `AD-ICICIB`, `BZ-SBIINB`), multi-modal transaction channels (UPI, IMPS, NEFT, ATM, POS, NACH), varied currency representations (`Rs.`, `INR`, `₹`), masked account numbers (`**4821`, `XX1042`), OTP codes, bills, and rampant spam/promotional noise.

### 🛠️ Architecture & Workflow
```mermaid
flowchart LR
    A[Raw Indian SMS] --> B[Synthetic Corpus Generator]
    B --> C[HF Chat Template]
    C --> D[Qwen Base Model]
    D --> E[PEFT LoRA Adapters]
    E --> F[TRL SFTTrainer]
    F --> G[Fine-Tuned Adapter / Merged Model]
    G --> H[FastAPI Microservice / App]
```


## 1. Environment & Hardware Diagnostics
Check PyTorch device (CUDA, Apple Silicon MPS, or CPU), available memory, and core library versions.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import datasets
import peft
import pydantic
import torch
import transformers
import trl

print(f"Torch version:        {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT version:         {peft.__version__}")
print(f"TRL version:          {trl.__version__}")
print(f"Datasets version:     {datasets.__version__}")
print(f"Pydantic version:     {pydantic.__version__}")

# Compute device diagnostics
if torch.cuda.is_available():
    device = "cuda"
    print(f"✅ CUDA Device detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("✅ Apple Silicon MPS (Metal Performance Shaders) detected")
else:
    device = "cpu"
    print("ℹ️ Running on CPU (training batch size should be kept small or run on GPU/MPS)")

## 2. Indian SMS Dataset Generation
We use the modular `IndianSMSGenerator` from `slm_sms.data` to synthesize realistic financial messages covering:
- **Debit:** UPI transactions, ATM withdrawals, POS card swipes, NetBanking, Auto-debit/EMI
- **Credit / Refund:** Salary credits, UPI inward payments, merchant refunds, cashbacks
- **OTP:** Bank 3D-secure OTP, login codes, Aadhaar e-KYC
- **Bills:** Credit card statement dues, electricity bills, telecom postpaid
- **Spam / Promo:** Instant loan phishing, casino/rummy rewards, fake KYC threats

In [ ]:
from slm_sms.data.dataset import save_jsonl
from slm_sms.data.generator import IndianSMSGenerator

generator = IndianSMSGenerator(seed=42)
data_splits = generator.generate_dataset(n_samples=2000, val_split=0.15, test_split=0.05)

print(f"Generated Train samples:      {len(data_splits['train'])}")
print(f"Generated Validation samples: {len(data_splits['validation'])}")
print(f"Generated Test samples:       {len(data_splits['test'])}")

# Save to local data directory
data_dir = project_root / "data"
save_jsonl(data_splits["train"], data_dir / "train.jsonl")
save_jsonl(data_splits["validation"], data_dir / "val.jsonl")
save_jsonl(data_splits["test"], data_dir / "test.jsonl")
print(f"Saved dataset splits to {data_dir}/")

### Inspect Generated Samples

In [ ]:
import json

# Inspect 5 sample messages
for i in range(5):
    sample = data_splits["train"][i]["messages"]
    print(f"=== Sample {i + 1} ===")
    print(f"System:    {sample[0]['content']}")
    print(f"User (SMS): {sample[1]['content']}")
    print(f"Assistant:  {sample[2]['content']}")
    print()

## 3. Load Qwen Model and Tokenizer
We configure the base model and examine its zero-shot capabilities.

In [ ]:
from slm_sms.models.model_loader import (
    get_lora_config,
    load_tokenizer,
)

# Target model (Change to Qwen/Qwen3.5-2B or Qwen/Qwen2.5-1.5B-Instruct)
MODEL_ID = "Qwen/Qwen3.5-2B"

tokenizer = load_tokenizer(MODEL_ID)
print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}, Pad token: {tokenizer.pad_token}")

### Test Zero-Shot Base Model Output
Let's see how the base model behaves before fine-tuning.

In [ ]:
test_sms = "[VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50."

prompt_messages = [
    {"role": "system", "content": "Extract structured JSON from SMS. Output valid JSON only."},
    {"role": "user", "content": test_sms},
]

formatted = tokenizer.apply_chat_template(
    prompt_messages, tokenize=False, add_generation_prompt=True
)
print("Formatted chat template prompt:")
print(formatted)

## 4. Setup PEFT LoRA (Low-Rank Adaptation)
Rather than updating all 2B parameters, LoRA freezes the backbone weights and injects small trainable rank decomposition matrices into the attention and MLP feed-forward projections (`q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`).

This reduces trainable parameters by **~99%**, cutting VRAM requirements and allowing fast, efficient convergence.

In [ ]:
lora_config = get_lora_config(r=16, lora_alpha=32, lora_dropout=0.05)
print("LoRA Configuration:")
print(lora_config)

## 5. Fine-Tuning with TRL SFTTrainer
We initialize the `SMSTrainingPipeline` to execute supervised instruction fine-tuning.

In [ ]:
from slm_sms.data.dataset import load_hf_dataset_from_dir
from slm_sms.training.trainer import SMSTrainingPipeline

output_dir = project_root / "output" / "qwen-sms-lora"
hf_dataset = load_hf_dataset_from_dir(data_dir)

pipeline = SMSTrainingPipeline(
    model_id=MODEL_ID,
    output_dir=output_dir,
    lora_r=16,
    lora_alpha=32,
    use_4bit=False,  # Set to True if on CUDA with bitsandbytes
)

# To run fine-tuning:
# metrics = pipeline.train(
#     dataset=hf_dataset,
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
# )
# print("Training metrics:", metrics)

## 6. Evaluation & Accuracy Benchmarking
Evaluate predictions on the holdout test set with `compute_extraction_metrics`.
We test:
1. **JSON Validity Rate** (Strict JSON parser check)
2. **Type Accuracy** (debit, credit, otp, bill, spam)
3. **Amount Exact Match Rate**
4. **Merchant & Account Accuracy**
5. **Spam Detection Precision, Recall, & F1**

In [ ]:
from slm_sms.training.metrics import compute_extraction_metrics

# Synthetic evaluation demonstration
sample_predictions = [
    '{"type":"debit","amount":1450.0,"currency":"INR","merchant":"swiggy@axisbank","account":"4821","spam":false}',
    '{"type":"otp","amount":2499.0,"currency":"INR","merchant":"ZOMATO","account":"7712","otp":"489201","spam":false}',
    '{"type":"spam","amount":500000.0,"currency":"INR","spam":true}',
]

sample_ground_truths = [
    {
        "type": "debit",
        "amount": 1450.0,
        "currency": "INR",
        "merchant": "swiggy@axisbank",
        "account": "4821",
        "spam": False,
    },
    {
        "type": "otp",
        "amount": 2499.0,
        "currency": "INR",
        "merchant": "ZOMATO",
        "account": "7712",
        "otp": "489201",
        "spam": False,
    },
    {"type": "spam", "amount": 500000.0, "currency": "INR", "spam": True},
]

report = compute_extraction_metrics(sample_predictions, sample_ground_truths)
print("Evaluation Report:")
for k, v in report.to_dict().items():
    print(f"  {k:22s}: {v}")

## 7. Interactive SMS Entity Extraction Widget
Test any Indian SMS below using the inference engine.

In [ ]:
from slm_sms.inference.engine import SMSExtractorEngine

# Example SMS queries
test_queries = [
    "[VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50.",
    "489201 is your secret OTP for transaction of Rs.2,499.00 on your Axis Bank Card **7712 at ZOMATO. Valid for 5 mins. Do not share OTP.",
    "[BZ-SBIINB] A/c *8921 credited by Rs.75,000.00 on 23-Nov-26 by Salary transfer from TECH CORP PVT LTD. Avl Bal: Rs.87,400.00.",
    "Congratulations! Pre-approved personal loan of Rs.5,00,000 without income proof. Click bit.ly/loan-instant to claim before midnight!",
]

print("Testing Fallback & Schema Parser on diverse inputs:")
engine_stub = SMSExtractorEngine.__new__(SMSExtractorEngine)
for query in test_queries:
    entity = engine_stub._fallback_regex_extraction(query)
    print(f"\nInput SMS: {query}")
    print("Parsed JSON:")
    print(json.dumps(entity.to_dict(), indent=2))

## 8. Export Merged Model for Production Deployment
To deploy with zero adapter overhead in production (e.g. via **vLLM**, **Triton**, or **FastAPI**), merge the LoRA adapter back into base weights:

```bash
# Run from terminal:
uv run python scripts/export_merged_model.py \
  --base_model Qwen/Qwen3.5-2B \
  --adapter_path ./output/qwen-sms-lora \
  --output_dir ./models/qwen-sms-merged
```

Then launch the production FastAPI microservice:
```bash
uv run python scripts/serve.py --model_id ./models/qwen-sms-merged --port 8000
```
